#DebugV1.0

In [2]:
"""
Healthcare Professional Analytics Pipeline - FULLY FIXED VERSION
Complete system for scoring and clustering doctors based on 4 personality buckets
Handles encoding issues, missing columns, and digital data processing

Author: Healthcare Analytics Team
Version: 1.2 - Fixed digital data processing and output completeness
"""

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
import chardet
warnings.filterwarnings('ignore')

# =====================================================
# CONFIGURATION
# =====================================================

# File paths
DATA_DIR = r'C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\Model_Data'

# Sample file names
FILES = {
    # Engagement data (Jan-July 2025)
    'engagement': [
        'Jan25.csv', 'Feb25.csv', 'March25.csv',
        'April25.csv', 'May25.csv', 'June25.csv', 'July25.csv'
    ],
    
    # Professional info
    'pi': 'pi.csv',
    'clinics': 'clinics.csv',
    
    # Academic/Research data
    'publications': 'Publications.csv',
    'trials': 'Trials.csv',
    'academic_associations': 'Academic_association.csv',
    
    # Social Media data
    'digital': 'digital.csv',
    'healthcare_platforms': 'Healthcare.csv',
    
    # Recognition data
    'awards': 'awards.csv',
    'press': 'press.csv',
    'events': 'events.csv',
    'associations': 'association.csv'
}

# Clustering eligibility rules
ELIGIBILITY_MODE = 'Basic'  # 'strict', 'moderate', 'relaxed or 'basic'

ELIGIBILITY_RULES = {
    'strict': {'min_buckets': 4, 'engagement_required': False},
    'moderate': {'min_buckets': 3, 'engagement_required': False},
    'relaxed': {'min_buckets': 2, 'engagement_required': False},
    'Basic': {'min_buckets': 1, 'engagement_required': False}
}

# Scoring configuration
SCORING_CONFIG = {
    'engagement': {
        'email_open_weight': 0.50,
        'email_click_weight': 0.50,
        'whatsapp_read_weight': 0.50,
        'whatsapp_click_weight': 0.50,
        'call_productive_weight': 0.70,
        'call_duration_weight': 0.30,
        'final_email_weight': 0.33,
        'final_whatsapp_weight': 0.33,
        'final_call_weight': 0.34
    }
}

# =====================================================
# ENHANCED FILE LOADING WITH ENCODING DETECTION
# =====================================================

def detect_encoding(file_path):
    """Detect file encoding using chardet"""
    try:
        with open(file_path, 'rb') as file:
            raw_data = file.read(100000)
            result = chardet.detect(raw_data)
            encoding = result['encoding']
            confidence = result['confidence']
            print(f"      Detected encoding: {encoding} (confidence: {confidence:.2f})")
            return encoding
    except Exception as e:
        print(f"      Could not detect encoding: {e}")
        return None

def load_csv_with_encoding_detection(file_path):
    """Try to load CSV with multiple encoding strategies"""
    encodings_to_try = []
    
    detected = detect_encoding(file_path)
    if detected:
        encodings_to_try.append(detected)
    
    encodings_to_try.extend(['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'windows-1252'])
    
    for encoding in encodings_to_try:
        try:
            df = pd.read_csv(file_path, encoding=encoding, low_memory=False, on_bad_lines='skip')
            print(f"      Successfully loaded with {encoding} encoding")
            return df
        except (UnicodeDecodeError, LookupError):
            continue
        except Exception as e:
            print(f"      Error with {encoding}: {e}")
            continue
    
    try:
        df = pd.read_csv(file_path, encoding='utf-8', errors='ignore', low_memory=False, on_bad_lines='skip')
        print(f"      Loaded with UTF-8 (ignoring errors)")
        return df
    except Exception as e:
        print(f"      Failed to load file: {e}")
        return pd.DataFrame()

# =====================================================
# STEP 1: DATA LOADING & MERGING
# =====================================================

def load_engagement_data():
    """Load and average engagement data from Jan-July"""
    print("\n📊 Loading Engagement Data (Jan-July 2025)...")
    
    engagement_dfs = []
    total_rows_loaded = 0
    
    for file in FILES['engagement']:
        try:
            file_path = f"{DATA_DIR}\\{file}"
            print(f"   Loading {file}...")
            df = load_csv_with_encoding_detection(file_path)
            if not df.empty:
                engagement_dfs.append(df)
                total_rows_loaded += len(df)
                print(f"   ✓ Loaded {file}: {len(df)} rows, {df['UIN'].nunique()} unique UINs")
            else:
                print(f"   ⚠ File is empty: {file}")
        except FileNotFoundError:
            print(f"   ⚠ File not found: {file}")
        except Exception as e:
            print(f"   ❌ Error loading {file}: {e}")
    
    if not engagement_dfs:
        print("   ❌ No engagement data loaded!")
        return pd.DataFrame()
    
    print(f"\n   📊 Total rows before combining: {total_rows_loaded}")
    combined = pd.concat(engagement_dfs, ignore_index=True)
    print(f"   📊 Combined dataframe: {len(combined)} rows")
    print(f"   📊 Unique UINs in combined: {combined['UIN'].nunique()}")
    
    # CRITICAL FIX: Standardize column names (remove spaces, convert to lowercase with underscores)
    # BUT preserve UIN as uppercase
    column_mapping = {}
    for col in combined.columns:
        if col == 'UIN':
            column_mapping[col] = 'UIN'  # Keep UIN uppercase
        else:
            column_mapping[col] = col.strip().lower().replace(' ', '_')
    
    combined = combined.rename(columns=column_mapping)
    print(f"   🔍 DEBUG - Columns after standardization: {combined.columns.tolist()}")
    
    expected_cols = {
        'hcp_email_open_rate': 0,
        'hcp_email_click_rate': 0,
        'hcp_whatsapp_read_rate': 0,
        'hcp_whatsapp_click_rate': 0,
        'hcp_call_productive_rate': 0,
        'average_duration_connected_calls': 0
    }
    
    for col, default in expected_cols.items():
        if col not in combined.columns:
            combined[col] = default
            print(f"   ⚠ Column '{col}' not found, using default value: {default}")
        else:
            # Convert to numeric, handling any non-numeric values
            combined[col] = pd.to_numeric(combined[col], errors='coerce').fillna(default)
    
    agg_dict = {col: 'mean' for col in expected_cols.keys() if col in combined.columns}
    engagement_avg = combined.groupby('UIN', as_index=False).agg(agg_dict)
    
    print(f"   ✓ Averaged engagement data: {len(engagement_avg)} unique doctors")
    print(f"   🔍 DEBUG - UIN sample after averaging: {engagement_avg['UIN'].head(10).tolist()}")
    
    return engagement_avg

def load_csv_safe(file_key):
    """Safely load a CSV file with encoding detection"""
    filename = FILES[file_key]
    file_path = f"{DATA_DIR}\\{filename}"
    
    try:
        print(f"   Loading {file_key}...")
        df = load_csv_with_encoding_detection(file_path)
        if not df.empty and 'UIN' in df.columns:
            print(f"   ✓ {file_key}: {len(df)} rows, {len(df['UIN'].unique())} unique UINs")
            return df
        elif df.empty:
            print(f"   ⚠ {file_key} is empty")
            return pd.DataFrame()
        else:
            print(f"   ⚠ {file_key} loaded but missing 'UIN' column")
            return pd.DataFrame()
    except FileNotFoundError:
        print(f"   ⚠ {file_key} not found: {filename}")
        return pd.DataFrame()
    except Exception as e:
        print(f"   ❌ Error loading {file_key}: {e}")
        return pd.DataFrame()

def load_all_data():
    """Load all CSV files"""
    print("\n📁 Loading All Data Files...")
    
    data = {}
    
    data['engagement'] = load_engagement_data()
    data['pi'] = load_csv_safe('pi')
    data['clinics'] = load_csv_safe('clinics')
    data['publications'] = load_csv_safe('publications')
    data['trials'] = load_csv_safe('trials')
    data['academic_associations'] = load_csv_safe('academic_associations')
    data['digital'] = load_csv_safe('digital')
    data['healthcare_platforms'] = load_csv_safe('healthcare_platforms')
    data['awards'] = load_csv_safe('awards')
    data['press'] = load_csv_safe('press')
    data['events'] = load_csv_safe('events')
    data['associations'] = load_csv_safe('associations')
    
    return data


# =====================================================
# STEP 2: DATA AGGREGATION WITH DATA VALIDATION
# =====================================================

def aggregate_data_by_uin(data):
    """Aggregate all data sources by UIN - counting DISTINCT values, not rows"""
    print("\n🔗 Aggregating Data by UIN...")
    
    if data['pi'].empty:
        print("   ❌ No PI data available - cannot proceed")
        return pd.DataFrame()
    
    master = data['pi'][['UIN']].drop_duplicates().copy()
    print(f"   Master list: {len(master)} UINs from PI data (all doctors)")
    
    # Publications - Count distinct publications
    if not data['publications'].empty and 'UIN' in data['publications'].columns:
        print("\n   📚 Processing Publications:")
        print(f"      Raw rows: {len(data['publications'])}")
        
        pubs_clean = data['publications'][data['publications']['UIN'].notna()].copy()
        
        # Count distinct journals or publication IDs
        if 'publication_id' in pubs_clean.columns:
            pubs = pubs_clean.groupby('UIN').agg(
                publication_count=('publication_id', 'nunique')  # Count unique publication IDs
            ).reset_index()
            print(f"      Counting distinct publication_ids")
        elif 'published_title' in pubs_clean.columns:
            # Filter out empty titles
            pubs_clean = pubs_clean[
                (pubs_clean['published_title'].notna()) & 
                (pubs_clean['published_title'] != '')
            ]
            pubs = pubs_clean.groupby('UIN').agg(
                publication_count=('published_title', 'nunique')  # Count unique titles
            ).reset_index()
            print(f"      Counting distinct published_titles")
        elif 'journal' in pubs_clean.columns:
            pubs_clean = pubs_clean[
                (pubs_clean['journal'].notna()) & 
                (pubs_clean['journal'] != '')
            ]
            pubs = pubs_clean.groupby('UIN').agg(
                publication_count=('journal', 'nunique')  # Count unique journals
            ).reset_index()
            print(f"      Counting distinct journals")
        else:
            # Fallback to row count
            pubs = pubs_clean.groupby('UIN').agg(
                publication_count=('UIN', 'count')
            ).reset_index()
            print(f"      Using row count (no unique identifier found)")
        
        master = master.merge(pubs, on='UIN', how='left')
        print(f"      ✓ {len(pubs)} doctors have publications")
        if len(pubs) > 0:
            print(f"      Distribution: min={pubs['publication_count'].min()}, max={pubs['publication_count'].max()}, mean={pubs['publication_count'].mean():.2f}")
    
    # Trials - Count distinct trials
    if not data['trials'].empty and 'UIN' in data['trials'].columns:
        print("\n   🔬 Processing Trials:")
        print(f"      Raw rows: {len(data['trials'])}")
        
        trials_clean = data['trials'][data['trials']['UIN'].notna()].copy()
        
        if 'trial_id' in trials_clean.columns:
            # Filter out empty/invalid trial IDs
            trials_clean = trials_clean[
                (trials_clean['trial_id'].notna()) & 
                (trials_clean['trial_id'] != '') &
                (trials_clean['trial_id'].str.upper() != 'NA')
            ]
            trials = trials_clean.groupby('UIN').agg(
                trial_count=('trial_id', 'nunique')  # Count unique trial IDs
            ).reset_index()
            print(f"      Counting distinct trial_ids")
        elif 'trial_public_title' in trials_clean.columns:
            trials_clean = trials_clean[
                (trials_clean['trial_public_title'].notna()) & 
                (trials_clean['trial_public_title'] != '')
            ]
            trials = trials_clean.groupby('UIN').agg(
                trial_count=('trial_public_title', 'nunique')  # Count unique titles
            ).reset_index()
            print(f"      Counting distinct trial titles")
        else:
            trials = trials_clean.groupby('UIN').agg(
                trial_count=('UIN', 'count')
            ).reset_index()
            print(f"      Using row count")
        
        master = master.merge(trials, on='UIN', how='left')
        print(f"      ✓ {len(trials)} doctors have trials")
    
    # Awards - Count distinct awards
    if not data['awards'].empty and 'UIN' in data['awards'].columns:
        print("\n   🏆 Processing Awards:")
        print(f"      Raw rows: {len(data['awards'])}")
        
        awards_clean = data['awards'][data['awards']['UIN'].notna()].copy()
        
        if 'award_name' in awards_clean.columns:
            awards_clean = awards_clean[
                (awards_clean['award_name'].notna()) & 
                (awards_clean['award_name'] != '') &
                (awards_clean['award_name'].str.upper() != 'NA')
            ]
            awards = awards_clean.groupby('UIN').agg(
                award_count=('award_name', 'nunique')  # Count unique award names
            ).reset_index()
            print(f"      Counting distinct award_names")
        elif 'award_id' in awards_clean.columns:
            awards_clean = awards_clean[
                (awards_clean['award_id'].notna()) & 
                (awards_clean['award_id'] != '')
            ]
            awards = awards_clean.groupby('UIN').agg(
                award_count=('award_id', 'nunique')
            ).reset_index()
            print(f"      Counting distinct award_ids")
        else:
            awards = awards_clean.groupby('UIN').agg(
                award_count=('UIN', 'count')
            ).reset_index()
            print(f"      Using row count")
        
        master = master.merge(awards, on='UIN', how='left')
        print(f"      ✓ {len(awards)} doctors have awards")
    
    # Digital Presence - Count distinct platforms
    if not data['digital'].empty and 'UIN' in data['digital'].columns:
        print("\n   📱 Processing Digital/Social Media:")
        print(f"      Raw rows: {len(data['digital'])}")
        
        digital_clean = data['digital'][
            (data['digital']['UIN'].notna()) & 
            (data['digital']['UIN'] != '')
        ].copy()
        
        # Find platform column
        platform_col = None
        for col in ['platform', 'platform_name', 'social_media', 'channel', 'sm_platform']:
            if col in data['digital'].columns:
                platform_col = col
                break
        
        if platform_col:
            print(f"      Found platform column: '{platform_col}'")
            # Filter out empty platforms
            digital_clean = digital_clean[
                (digital_clean[platform_col].notna()) & 
                (digital_clean[platform_col] != '') &
                (digital_clean[platform_col].str.upper() != 'NA')
            ]
            
            # Count distinct platforms per UIN
            digital = digital_clean.groupby('UIN').agg(
                platform_count=(platform_col, 'nunique')  # Count unique platforms
            ).reset_index()
            print(f"      Counting distinct platforms")
            
            # Also aggregate followers if available
            if 'sm_followers' in digital_clean.columns:
                digital_clean['sm_followers'] = pd.to_numeric(
                    digital_clean['sm_followers'], errors='coerce'
                ).fillna(0)
                followers = digital_clean.groupby('UIN').agg(
                    total_followers=('sm_followers', 'sum')
                ).reset_index()
                digital = digital.merge(followers, on='UIN', how='left')
        else:
            print(f"      ⚠ No platform column found - using row count")
            digital = digital_clean.groupby('UIN').agg(
                platform_count=('UIN', 'count')
            ).reset_index()
        
        master = master.merge(digital, on='UIN', how='left')
        print(f"      ✓ {len(digital)} doctors have social media presence")
        
        # Show distribution - FIXED: removed indent parameter
        if len(digital) > 0:
            platform_dist = digital['platform_count'].value_counts().head(10)
            print(f"      Platform count distribution:")
            for val, count in platform_dist.items():
                print(f"          {val}: {count}")
    
    # Press - Count distinct articles/publications
    if not data['press'].empty and 'UIN' in data['press'].columns:
        print("\n   📰 Processing Press:")
        print(f"      Raw rows: {len(data['press'])}")
        
        press_clean = data['press'][data['press']['UIN'].notna()].copy()
        
        # Find content column
        content_col = None
        for col in ['article_title', 'press_title', 'headline', 'title', 'publication', 'press_name', 'press_id']:
            if col in data['press'].columns:
                content_col = col
                break
        
        if content_col:
            print(f"      Found content column: '{content_col}'")
            # Filter out empty content
            press_clean = press_clean[
                (press_clean[content_col].notna()) & 
                (press_clean[content_col] != '') &
                (press_clean[content_col].str.upper() != 'NA')
            ]
            
            # Count distinct articles/press mentions
            press = press_clean.groupby('UIN').agg(
                press_count=(content_col, 'nunique')  # Count unique articles
            ).reset_index()
            print(f"      Counting distinct {content_col}")
        else:
            print(f"      ⚠ No content column found - using row count")
            press = press_clean.groupby('UIN').agg(
                press_count=('UIN', 'count')
            ).reset_index()
        
        master = master.merge(press, on='UIN', how='left')
        print(f"      ✓ {len(press)} doctors have press mentions")
        
        # Show distribution - FIXED: removed indent parameter
        if len(press) > 0:
            press_dist = press['press_count'].value_counts().head(10)
            print(f"      Press count distribution:")
            for val, count in press_dist.items():
                print(f"          {val}: {count}")
    
    # Events - Count distinct events
    if not data['events'].empty and 'UIN' in data['events'].columns:
        print("\n   🎤 Processing Events:")
        print(f"      Raw rows: {len(data['events'])}")
        
        events_clean = data['events'][data['events']['UIN'].notna()].copy()
        
        if 'event_name' in events_clean.columns:
            events_clean = events_clean[
                (events_clean['event_name'].notna()) & 
                (events_clean['event_name'] != '') &
                (events_clean['event_name'].str.upper() != 'NA')
            ]
            events = events_clean.groupby('UIN').agg(
                event_count=('event_name', 'nunique')  # Count unique event names
            ).reset_index()
            print(f"      Counting distinct event_names")
        elif 'event_id' in events_clean.columns:
            events_clean = events_clean[
                (events_clean['event_id'].notna()) & 
                (events_clean['event_id'] != '')
            ]
            events = events_clean.groupby('UIN').agg(
                event_count=('event_id', 'nunique')
            ).reset_index()
            print(f"      Counting distinct event_ids")
        else:
            events = events_clean.groupby('UIN').agg(
                event_count=('UIN', 'count')
            ).reset_index()
            print(f"      Using row count")
        
        master = master.merge(events, on='UIN', how='left')
        print(f"      ✓ {len(events)} doctors have events")
        
        # Show distribution - FIXED
        if len(events) > 0:
            event_dist = events['event_count'].value_counts().head(10)
            print(f"      Event count distribution:")
            for val, count in event_dist.items():
                print(f"          {val}: {count}")
    
    # Professional Associations - Count distinct associations
    if not data['associations'].empty and 'UIN' in data['associations'].columns:
        print("\n   🤝 Processing Professional Associations:")
        print(f"      Raw rows: {len(data['associations'])}")
        
        assoc_clean = data['associations'][data['associations']['UIN'].notna()].copy()
        
        if 'association_name' in assoc_clean.columns:
            assoc_clean = assoc_clean[
                (assoc_clean['association_name'].notna()) & 
                (assoc_clean['association_name'] != '') &
                (assoc_clean['association_name'].str.upper() != 'NA')
            ]
            assoc = assoc_clean.groupby('UIN').agg(
                association_count=('association_name', 'nunique')  # Count unique associations
            ).reset_index()
            print(f"      Counting distinct association_names")
        elif 'association_id' in assoc_clean.columns:
            assoc_clean = assoc_clean[
                (assoc_clean['association_id'].notna()) & 
                (assoc_clean['association_id'] != '')
            ]
            assoc = assoc_clean.groupby('UIN').agg(
                association_count=('association_id', 'nunique')
            ).reset_index()
            print(f"      Counting distinct association_ids")
        else:
            assoc = assoc_clean.groupby('UIN').agg(
                association_count=('UIN', 'count')
            ).reset_index()
            print(f"      Using row count")
        
        master = master.merge(assoc, on='UIN', how='left')
        print(f"      ✓ {len(assoc)} doctors have professional associations")
        
        # Show distribution - FIXED
        if len(assoc) > 0:
            assoc_dist = assoc['association_count'].value_counts().head(10)
            print(f"      Association count distribution:")
            for val, count in assoc_dist.items():
                print(f"          {val}: {count}")
    
    # Fill NaN with 0 for count columns
    count_columns = ['publication_count', 'trial_count', 'academic_association_count',
                     'award_count', 'platform_count', 'total_followers', 
                     'healthcare_platform_count', 'press_count', 'event_count',
                     'association_count']
    
    for col in count_columns:
        if col in master.columns:
            master[col] = master[col].fillna(0)
        else:
            master[col] = 0
    
    # Add engagement data with proper NaN handling
    if not data['engagement'].empty:
        merged = master.merge(data['engagement'], on='UIN', how='left')
        
        # Fill NaN with 0 for all engagement columns
        engagement_cols = ['hcp_email_open_rate', 'hcp_email_click_rate', 
                          'hcp_whatsapp_read_rate', 'hcp_whatsapp_click_rate', 
                          'hcp_call_productive_rate', 'average_duration_connected_calls']
        for col in engagement_cols:
            if col in merged.columns:
                merged[col] = merged[col].fillna(0)
        
        print(f"\n   ✓ Merged engagement data")
    else:
        merged = master.copy()
        for col in ['hcp_email_open_rate', 'hcp_email_click_rate', 'hcp_whatsapp_read_rate',
                   'hcp_whatsapp_click_rate', 'hcp_call_productive_rate', 'average_duration_connected_calls']:
            merged[col] = 0
    
    # Add PI and clinic data
    if not data['pi'].empty and 'UIN' in data['pi'].columns:
        available_cols = ['UIN']
        for col in ['full_name', 'mobile', 'whatsapp_phone', 'email_id_1', 'specialty']:
            if col in data['pi'].columns:
                available_cols.append(col)
        
        if len(available_cols) > 1:
            pi_subset = data['pi'][available_cols].drop_duplicates('UIN')
            merged = merged.merge(pi_subset, on='UIN', how='left')
    
    if not data['clinics'].empty and 'UIN' in data['clinics'].columns:
        available_cols = ['UIN']
        for col in ['clinic_address', 'clinic_city', 'clinic_state']:
            if col in data['clinics'].columns:
                available_cols.append(col)
        
        if len(available_cols) > 1:
            clinic_subset = data['clinics'][available_cols].drop_duplicates('UIN')
            merged = merged.merge(clinic_subset, on='UIN', how='left')
    
    # Print final summary
    print(f"\n   📊 Final Data Summary:")
    print(f"   Total doctors: {len(merged)}")
    print(f"   Doctors with publications: {(merged['publication_count'] > 0).sum()}")
    print(f"   Doctors with trials: {(merged['trial_count'] > 0).sum()}")
    print(f"   Doctors with awards: {(merged['award_count'] > 0).sum()}")
    print(f"   Doctors with social media: {(merged['platform_count'] > 0).sum()}")
    print(f"   Doctors with press mentions: {(merged['press_count'] > 0).sum()}")
    print(f"   Doctors with events: {(merged['event_count'] > 0).sum()}")
    print(f"   Doctors with professional associations: {(merged['association_count'] > 0).sum()}")
    
    return merged

# =====================================================
# STEP 3: SCORING
# =====================================================

def calculate_engagement_score(row, config):
    """Calculate engagement bucket score"""
    email_open = row.get('hcp_email_open_rate', 0) or 0
    email_click = row.get('hcp_email_click_rate', 0) or 0
    email_score = (email_open * config['email_open_weight'] + 
                   email_click * config['email_click_weight'])
    
    wa_read = row.get('hcp_whatsapp_read_rate', 0) or 0
    wa_click = row.get('hcp_whatsapp_click_rate', 0) or 0
    wa_score = (wa_read * config['whatsapp_read_weight'] + 
                wa_click * config['whatsapp_click_weight'])
    
    call_prod = row.get('hcp_call_productive_rate', 0) or 0
    call_dur = row.get('average_duration_connected_calls', 0) or 0
    call_dur_norm = min((call_dur / 60) * 100, 100) if call_dur > 0 else 0
    call_score = (call_prod * config['call_productive_weight'] + 
                  call_dur_norm * config['call_duration_weight'])
    
    final = (email_score * config['final_email_weight'] + 
             wa_score * config['final_whatsapp_weight'] + 
             call_score * config['final_call_weight'])
    
    return round(final, 2)

def calculate_academic_score(row):
    """Calculate academic/research bucket score"""
    pubs = row.get('publication_count', 0) or 0
    trials = row.get('trial_count', 0) or 0
    acad_assoc = row.get('academic_association_count', 0) or 0
    
    pub_score = min(pubs * 5, 40)
    trial_score = min(trials * 15, 30)
    assoc_score = min(acad_assoc * 10, 30)
    
    total = pub_score + trial_score + assoc_score
    return round(min(total, 100), 2)

def calculate_social_media_score(row):
    """Calculate social media inclination bucket score"""
    platforms = row.get('platform_count', 0) or 0
    followers = row.get('total_followers', 0) or 0
    hc_platforms = row.get('healthcare_platform_count', 0) or 0
    
    if followers > 0:
        follower_score = min(np.log10(followers) * 10, 50)
    else:
        follower_score = 0
    
    platform_score = min(platforms * 10, 30)
    hc_score = min(hc_platforms * 10, 20)
    
    total = follower_score + platform_score + hc_score
    return round(min(total, 100), 2)

def calculate_recognition_score(row):
    """Calculate recognition bucket score"""
    awards = row.get('award_count', 0) or 0
    press = row.get('press_count', 0) or 0
    events = row.get('event_count', 0) or 0
    assoc = row.get('association_count', 0) or 0
    
    award_score = min(awards * 15, 30)
    press_score = min(press * 10, 25)
    event_score = min(events * 8, 25)
    assoc_score = min(assoc * 5, 20)
    
    total = award_score + press_score + event_score + assoc_score
    return round(min(total, 100), 2)

def score_all_doctors(merged_data):
    """Calculate all 4 bucket scores for each doctor"""
    print("\n🎯 Calculating Scores for All 4 Buckets...")
    
    df = merged_data.copy()
    
    # Calculate scores
    config = SCORING_CONFIG['engagement']
    df['engagement_score'] = df.apply(lambda row: calculate_engagement_score(row, config), axis=1)
    df['academic_score'] = df.apply(calculate_academic_score, axis=1)
    df['social_media_score'] = df.apply(calculate_social_media_score, axis=1)
    df['recognition_score'] = df.apply(calculate_recognition_score, axis=1)
    
    # CRITICAL FIX: Ensure engagement_score is never NaN
    df['engagement_score'] = df['engagement_score'].fillna(0)
    
    # Rest of the function remains the same...
    df['engagement_data_available'] = df['engagement_score'] > 0
    # ... continue with rest of the code
    df['academic_data_available'] = (
        df.get('publication_count', 0).fillna(0) + 
        df.get('trial_count', 0).fillna(0) + 
        df.get('academic_association_count', 0).fillna(0)
    ) > 0
    df['social_media_data_available'] = (
        df.get('platform_count', 0).fillna(0) + 
        df.get('healthcare_platform_count', 0).fillna(0)
    ) > 0
    df['recognition_data_available'] = (
        df.get('award_count', 0).fillna(0) + 
        df.get('press_count', 0).fillna(0) + 
        df.get('event_count', 0).fillna(0) + 
        df.get('association_count', 0).fillna(0)) > 0
    
    df['buckets_with_data'] = (
        df['engagement_data_available'].astype(int) +
        df['academic_data_available'].astype(int) +
        df['social_media_data_available'].astype(int) +
        df['recognition_data_available'].astype(int)
    )
    
    print(f"   ✓ Scored {len(df)} doctors")
    print(f"   Average scores:")
    print(f"      Engagement: {df['engagement_score'].mean():.2f}")
    print(f"      Academic: {df['academic_score'].mean():.2f}")
    print(f"      Social Media: {df['social_media_score'].mean():.2f}")
    print(f"      Recognition: {df['recognition_score'].mean():.2f}")
    print(f"\n   Data Availability:")
    print(f"      Doctors with social media data: {df['social_media_data_available'].sum()}")
    print(f"      Total platforms tracked: {df['platform_count'].sum():.0f}")
    print(f"      Total followers tracked: {df['total_followers'].sum():.0f}")
    
    return df

# =====================================================
# STEP 4: CLUSTERING ELIGIBILITY
# =====================================================

def assess_clustering_eligibility(df):
    """Determine which doctors are eligible for clustering"""
    print("\n✅ Assessing Clustering Eligibility...")
    
    rules = ELIGIBILITY_RULES[ELIGIBILITY_MODE]
    
    df['eligible_for_clustering'] = (
        (df['buckets_with_data'] >= rules['min_buckets']) &
        (df['engagement_data_available'] if rules['engagement_required'] else True)
    )
    
    def get_ineligibility_reason(row):
        if row['eligible_for_clustering']:
            return ''
        reasons = []
        if not row['engagement_data_available']:
            reasons.append('No engagement data')
        if row['buckets_with_data'] < rules['min_buckets']:
            reasons.append(f"Only {int(row['buckets_with_data'])}/{rules['min_buckets']} buckets have data")
        return '; '.join(reasons)
    
    df['insufficient_data_reason'] = df.apply(get_ineligibility_reason, axis=1)
    
    eligible_count = df['eligible_for_clustering'].sum()
    ineligible_count = len(df) - eligible_count
    
    print(f"   Mode: {ELIGIBILITY_MODE}")
    print(f"   ✓ Eligible for clustering: {eligible_count} doctors")
    print(f"   ✗ Ineligible: {ineligible_count} doctors")
    
    print(f"\n   Bucket Data Availability:")
    print(f"      Engagement: {df['engagement_data_available'].sum()} doctors")
    print(f"      Academic: {df['academic_data_available'].sum()} doctors")
    print(f"      Social Media: {df['social_media_data_available'].sum()} doctors")
    print(f"      Recognition: {df['recognition_data_available'].sum()} doctors")
    
    return df

# =====================================================
# STEP 5: K-MEANS CLUSTERING
# =====================================================

def perform_clustering(df, n_clusters=5):
    """Perform K-Means clustering on eligible doctors"""
    print(f"\n🔍 Performing K-Means Clustering (k={n_clusters})...")
    
    eligible = df[df['eligible_for_clustering']].copy()
    
    if len(eligible) < n_clusters:
        print(f"   ❌ Not enough eligible doctors ({len(eligible)}) for {n_clusters} clusters")
        df['cluster_id'] = None
        df['cluster_name'] = None
        return df, []
    
    features = eligible[['engagement_score', 'academic_score', 'social_media_score', 'recognition_score']].values
    
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(features_scaled)
    
    if len(eligible) > n_clusters:
        sil_score = silhouette_score(features_scaled, clusters)
        print(f"   ✓ Clustering completed")
        print(f"   Silhouette Score: {sil_score:.3f}")
    else:
        print(f"   ✓ Clustering completed (too few samples for silhouette score)")
    
    eligible['cluster_id'] = clusters
    
    cluster_profiles = []
    for i in range(n_clusters):
        cluster_data = eligible[eligible['cluster_id'] == i]
        profile = {
            'cluster_id': i,
            'size': len(cluster_data),
            'avg_engagement': cluster_data['engagement_score'].mean(),
            'avg_academic': cluster_data['academic_score'].mean(),
            'avg_social_media': cluster_data['social_media_score'].mean(),
            'avg_recognition': cluster_data['recognition_score'].mean()
        }
        
        scores = [profile['avg_engagement'], profile['avg_academic'], 
                 profile['avg_social_media'], profile['avg_recognition']]
        traits = ['Engagement', 'Academic', 'Social Media', 'Recognition']
        dominant = traits[scores.index(max(scores))]
        profile['cluster_name'] = f"Cluster {i}: {dominant}-Focused"
        
        cluster_profiles.append(profile)
        print(f"   Cluster {i}: {len(cluster_data)} doctors - {profile['cluster_name']}")
    
    # Merge clusters back to main dataframe
    df = df.merge(eligible[['UIN', 'cluster_id']], on='UIN', how='left', suffixes=('', '_new'))
    if 'cluster_id_new' in df.columns:
        df['cluster_id'] = df['cluster_id_new']
        df.drop('cluster_id_new', axis=1, inplace=True)
    
    # Add cluster names
    cluster_map = {p['cluster_id']: p['cluster_name'] for p in cluster_profiles}
    df['cluster_name'] = df['cluster_id'].map(cluster_map)
    
    return df, cluster_profiles

# =====================================================
# STEP 6: OUTPUT GENERATION (FIXED - INCLUDES ALL ROWS!)
# =====================================================

def generate_output(df, cluster_profiles):
    """Generate final output files - INCLUDES ALL DOCTORS"""
    print("\n💾 Generating Output Files...")
    
    base_cols = ['UIN']
    optional_cols = [
        'full_name', 'specialty', 'mobile', 'whatsapp_phone', 'email_id_1',
        'clinic_address', 'clinic_city', 'clinic_state'
    ]
    required_cols = [
        'engagement_score', 'engagement_data_available',
        'academic_score', 'academic_data_available',
        'social_media_score', 'social_media_data_available',
        'recognition_score', 'recognition_data_available',
        'buckets_with_data', 'eligible_for_clustering', 
        'cluster_id', 'cluster_name', 'insufficient_data_reason'
    ]
    
    output_cols = base_cols.copy()
    
    for col in optional_cols:
        if col in df.columns:
            output_cols.append(col)
    
    output_cols.extend(required_cols)
    
    for col in output_cols:
        if col not in df.columns:
            df[col] = None
    
    output_df = df[output_cols].copy()
    
    # Save main output - ALL ROWS INCLUDED
    output_file = f"{DATA_DIR}\\scored_doctors_with_clusters.csv"
    try:
        output_df.to_csv(output_file, index=False, encoding='utf-8')
        print(f"   ✓ Saved: {output_file}")
    except Exception as e:
        print(f"   ❌ Error saving main file: {e}")
        try:
            output_df.to_csv(output_file, index=False, encoding='latin-1')
            print(f"   ✓ Saved with latin-1 encoding: {output_file}")
        except Exception as e2:
            print(f"   ❌ Could not save main file: {e2}")
    
    # Save cluster profiles
    if cluster_profiles:
        profiles_df = pd.DataFrame(cluster_profiles)
        profiles_file = f"{DATA_DIR}\\cluster_profiles.csv"
        try:
            profiles_df.to_csv(profiles_file, index=False)
            print(f"   ✓ Saved: {profiles_file}")
        except Exception as e:
            print(f"   ❌ Error saving profiles: {e}")
    
    # Summary statistics
    print(f"\n📊 Summary Statistics:")
    print(f"   Total doctors in output: {len(output_df)}")
    print(f"   Eligible for clustering: {output_df['eligible_for_clustering'].sum()}")
    print(f"   Clustered: {output_df['cluster_id'].notna().sum()}")
    print(f"   Ineligible (insufficient data): {(~output_df['eligible_for_clustering']).sum()}")
    
    # Breakdown by buckets
    print(f"\n   Bucket Distribution:")
    for i in range(5):
        count = (output_df['buckets_with_data'] == i).sum()
        print(f"      {i} buckets: {count} doctors")
    
    return output_df

# =====================================================
# MAIN EXECUTION
# =====================================================

def main():
    print("=" * 70)
    print("HEALTHCARE PROFESSIONAL ANALYTICS PIPELINE")
    print("4-Bucket Scoring + K-Means Clustering")
    print("Version 1.2 - FIXED: Digital data processing & complete output")
    print("=" * 70)
    
    try:
        # Step 1: Load data
        data = load_all_data()
        
        if data['engagement'].empty:
            print("\n❌ No engagement data loaded - cannot proceed with analysis")
            return
        
        # Step 2: Aggregate by UIN
        merged = aggregate_data_by_uin(data)
        
        if merged.empty:
            print("\n❌ No data to process after aggregation")
            return
        
        # Step 3: Calculate scores
        scored = score_all_doctors(merged)
        
        # Step 4: Assess eligibility
        assessed = assess_clustering_eligibility(scored)
        
        # Step 5: Cluster eligible doctors
        clustered, profiles = perform_clustering(assessed, n_clusters=5)
        
        # Step 6: Generate output
        final_output = generate_output(clustered, profiles)
        
        print("\n" + "=" * 70)
        print("✓ PIPELINE COMPLETED SUCCESSFULLY")
        print("=" * 70)
        print(f"\nOutput files created in: {DATA_DIR}")
        print("  1. scored_doctors_with_clusters.csv (ALL doctors included)")
        if profiles:
            print("  2. cluster_profiles.csv")
        
        print("\n🔍 Key Fixes Applied:")
        print("  ✓ Fixed digital data processing indentation")
        print("  ✓ Filtered out blank UINs in Digital.csv")
        print("  ✓ All 10,000 doctors now included in output")
        print("  ✓ Social media scores now calculated correctly")
            
    except Exception as e:
        print(f"\n❌ Pipeline failed with error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

HEALTHCARE PROFESSIONAL ANALYTICS PIPELINE
4-Bucket Scoring + K-Means Clustering
Version 1.2 - FIXED: Digital data processing & complete output

📁 Loading All Data Files...

📊 Loading Engagement Data (Jan-July 2025)...
   Loading Jan25.csv...
      Detected encoding: ascii (confidence: 1.00)
      Successfully loaded with ascii encoding
   ✓ Loaded Jan25.csv: 1048575 rows, 11570 unique UINs
   Loading Feb25.csv...
      Detected encoding: ascii (confidence: 1.00)
      Successfully loaded with ascii encoding
   ✓ Loaded Feb25.csv: 12954 rows, 12953 unique UINs
   Loading March25.csv...
      Detected encoding: ascii (confidence: 1.00)
      Successfully loaded with ascii encoding
   ✓ Loaded March25.csv: 13041 rows, 13040 unique UINs
   Loading April25.csv...
      Detected encoding: ascii (confidence: 1.00)
      Successfully loaded with ascii encoding
   ✓ Loaded April25.csv: 13797 rows, 13796 unique UINs
   Loading May25.csv...
      Detected encoding: ascii (confidence: 1.00)
     

In [1]:
print("Digital.csv columns:", data['digital'].columns.tolist())
print("Sample digital data:")
print(data['digital'].head(20))

NameError: name 'data' is not defined

In [6]:
def debug_digital_data(data):
    """Debug function to inspect digital data structure"""
    print("\n" + "="*70)
    print("🔍 DIGITAL DATA DEBUG")
    print("="*70)
    
    if data['digital'].empty:
        print("❌ Digital data is EMPTY!")
        return
    
    print(f"\n📊 Digital.csv Structure:")
    print(f"   Total rows: {len(data['digital'])}")
    print(f"   Unique UINs: {data['digital']['UIN'].nunique()}")
    print(f"   Columns: {data['digital'].columns.tolist()}")
    
    print(f"\n📋 Sample Data (first 20 rows):")
    print(data['digital'].head(20).to_string())
    
    # Check for platform-related columns
    print(f"\n🔍 Looking for platform columns:")
    platform_cols = [col for col in data['digital'].columns if 'platform' in col.lower() or 'channel' in col.lower()]
    print(f"   Found: {platform_cols}")
    
    if platform_cols:
        for col in platform_cols:
            print(f"\n   Column '{col}' - unique values:")
            print(f"      {data['digital'][col].value_counts().head(10)}")
    
    # Check sm_channel specifically
    if 'sm_channel' in data['digital'].columns:
        print(f"\n📱 sm_channel distribution:")
        print(data['digital']['sm_channel'].value_counts())
        
        # Sample UINs with their channels
        print(f"\n   Sample UINs with channels:")
        sample = data['digital'][['UIN', 'sm_channel']].head(30)
        print(sample.to_string())


def main():
    print("=" * 70)
    print("HEALTHCARE PROFESSIONAL ANALYTICS PIPELINE")
    print("=" * 70)
    
    try:
        # Step 1: Load data
        data = load_all_data()
        
        # ADD THIS DEBUG LINE:
        debug_digital_data(data)
        
        # Rest of your code...
        if data['engagement'].empty:
            print("\n❌ No engagement data loaded - cannot proceed")
            return
        

SyntaxError: incomplete input (1396702615.py, line 56)

In [4]:
def main():
    print("=" * 70)
    print("HEALTHCARE PROFESSIONAL ANALYTICS PIPELINE")
    print("=" * 70)
    
    try:
        # Step 1: Load data
        data = load_all_data()
        
        # ADD THIS DEBUG LINE:
        debug_digital_data(data)
        
        # Rest of your code...
        if data['engagement'].empty:
            print("\n❌ No engagement data loaded - cannot proceed")
            return
        

SyntaxError: incomplete input (3792400191.py, line 17)